# Entity Normalisation (NEN)

Normalises extracted entities to type-appropriate public ontologies -- no UMLS license required, all open/unauthenticated services:

| Entity type | Resource |
|---|---|
| `GeneOrGeneProduct` | NCBI Gene |
| `ChemicalEntity` | MeSH |
| `DiseaseOrPhenotypicFeature` | MeSH |
| `OrganismTaxon` | NCBI Taxonomy |
| `CellLine` | Cellosaurus |
| `SequenceVariant` | dbSNP (rsIDs) / NCBI Variation Services (best-effort; free-text HGVS notation without a transcript accession is marked unresolved rather than guessed) |

**Scope**: only entities that actually appear as a `source` or `target` in a relation are normalised -- entities NER found but that never participate in a relation would be isolated nodes in the KG anyway, so normalising them is wasted API calls.

**Runtime**: NCBI E-utilities allow ~3 requests/sec without an API key, ~10/sec with one. A free key takes 2 minutes to generate at [ncbi.nlm.nih.gov/account/settings](https://www.ncbi.nlm.nih.gov/account/settings/) (Settings -> API Key Management) and roughly **3x's the speed of this notebook** -- worth doing given the deadline. Results are cached to disk per `(type, text)` pair, so re-running this notebook after an interruption resumes rather than re-querying everything.

In [1]:
import sys
import json

import pandas as pd

from pathlib import Path

MODULES_PATH = Path("../modules").resolve()
if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

from config.data_config import DATA_CONFIG
from modules.entity_normalisation import normalize_entity, load_cache, save_cache
import modules.entity_normalisation as en_module

### Configuration

In [2]:
# ---------- Manual dataset split entry ----------
DATASET_SPLIT = "contemporary"

OUTPUT_DIRECTORY = DATA_CONFIG[DATASET_SPLIT]["output_directory"]
PREDICTIONS_PATH = f"{OUTPUT_DIRECTORY}{DATASET_SPLIT}_predictions.json"
CACHE_PATH = f"{OUTPUT_DIRECTORY}nen_cache/{DATASET_SPLIT}_nen_cache.json"

# ---------- NCBI API key (strongly recommended -- see markdown above) ----------
en_module.NCBI_EMAIL = None      # e.g. "you@university.ac.uk"
en_module.NCBI_API_KEY = None    # paste your key here if you have one
if en_module.NCBI_API_KEY:
    en_module._REQUEST_DELAY = 0.11  # ~9/sec, safely under the 10/sec limit

### Load Predictions

In [3]:
with open(PREDICTIONS_PATH, "r") as f:
    predictions = json.load(f)

entities_df = pd.DataFrame(predictions["entities"])
relations_df = pd.DataFrame(predictions["relations"])

print(f"Entities: {len(entities_df)}, Relations: {len(relations_df)}")

Entities: 2627, Relations: 1587


### Step 1-2: Scope Entities to Those Used in a Relation

Collect every unique `(pmid, text)` pair that appears as a `source` or `target`, then join against `entities_df` on `(pmid, text)` to attach each one's `type` -- needed to know which ontology to normalise it against.

In [4]:
needed_source = relations_df[["pmid", "source"]].rename(columns={"source": "text"})
needed_target = relations_df[["pmid", "target"]].rename(columns={"target": "text"})
needed = pd.concat([needed_source, needed_target]).drop_duplicates()

merged = needed.merge(entities_df, on=["pmid", "text"], how="left")

matched = merged[merged["type"].notna()].copy()
unmatched = merged[merged["type"].isna()].copy()

print(f"Relation endpoints needing normalisation: {len(needed)}")
print(f"  Matched to a known entity type: {len(matched)}")
print(f"  Unmatched (no corresponding NER extraction for this pmid+text): {len(unmatched)}")

if len(unmatched) > 0:
    print("\nUnmatched sample (these will be excluded from the KG -- no type means no ontology to normalise against):")
    print(unmatched.head(10))

Relation endpoints needing normalisation: 1614
  Matched to a known entity type: 1614
  Unmatched (no corresponding NER extraction for this pmid+text): 0


### Step 3: Deduplicate to Unique (text, type) Pairs

The same entity is often mentioned across many abstracts -- only normalise each unique `(text, type)` pair once, not once per PMID.

In [5]:
unique_to_normalize = matched[["text", "type"]].drop_duplicates().reset_index(drop=True)

print(f"Unique (text, type) pairs to normalise: {len(unique_to_normalize)}")
print()
print(unique_to_normalize["type"].value_counts())

# Rough runtime estimate (most types need 2 HTTP calls: esearch + esummary)
est_calls = len(unique_to_normalize) * 2
est_seconds = est_calls * en_module._REQUEST_DELAY
print(f"\nEstimated calls: ~{est_calls}, estimated runtime: ~{est_seconds / 60:.1f} minutes")

Unique (text, type) pairs to normalise: 1440

type
DiseaseOrPhenotypicFeature    575
GeneOrGeneProduct             419
ChemicalEntity                275
SequenceVariant               102
OrganismTaxon                  59
CellLine                       10
Name: count, dtype: int64

Estimated calls: ~2880, estimated runtime: ~16.3 minutes


### Step 4: Run Normalisation

Cached per `(type, text)` pair -- safe to interrupt and re-run.

In [6]:
cache = load_cache(CACHE_PATH)

results = []
for i, row in unique_to_normalize.iterrows():
    result = normalize_entity(row["text"], row["type"], cache=cache, cache_path=CACHE_PATH)
    results.append(result)

    if (i + 1) % 50 == 0 or (i + 1) == len(unique_to_normalize):
        print(f"\rNormalised {i + 1}/{len(unique_to_normalize)}", end="", flush=True)

print()

norm_df = unique_to_normalize.copy()
norm_df["normalized_id"] = [r["normalized_id"] for r in results]
norm_df["normalized_name"] = [r["normalized_name"] for r in results]
norm_df["source_ontology"] = [r["source"] for r in results]
norm_df["resolved"] = [r["resolved"] for r in results]
norm_df["note"] = [r["note"] for r in results]

Normalised 1440/1440


### Resolution Summary

Worth checking this per-type before moving on -- a low resolution rate on a specific type (e.g. `SequenceVariant`, per the caveat above) is expected; a low rate on `GeneOrGeneProduct` or `OrganismTaxon` likely signals a real problem worth investigating (e.g. wrong search term construction) rather than an ontology coverage gap.

In [7]:
resolution_summary = (
    norm_df.groupby("type")["resolved"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "resolved_count", "count": "total"})
)
resolution_summary["resolution_rate"] = (
    resolution_summary["resolved_count"] / resolution_summary["total"]
).round(3)

resolution_summary

,resolved_count,total,resolution_rate
type,,,
CellLine,0,10,0.000
ChemicalEntity,236,275,0.858
DiseaseOrPhenotypicFeature,425,575,0.739
GeneOrGeneProduct,403,419,0.962
OrganismTaxon,40,59,0.678
SequenceVariant,17,102,0.167


### Step 5: Build the Node Table

Deduplicate by `(type, normalized_id)` -- this is where normalisation actually consolidates entities (e.g. multiple surface forms of the same gene collapsing to one node). Only resolved entities become nodes; unresolved ones are excluded here but kept in `norm_df` above for inspection/appendix reporting.

In [8]:
resolved_norm = norm_df[norm_df["resolved"]].copy()

# attach the (pmid, text) rows so we can aggregate provenance per normalised entity
resolved_with_pmids = matched.merge(
    resolved_norm[["text", "type", "normalized_id", "normalized_name", "source_ontology"]],
    on=["text", "type"], how="inner"
)

nodes_df = (
    resolved_with_pmids
    .groupby(["type", "normalized_id"])
    .agg(
        normalized_name=("normalized_name", "first"),
        source_ontology=("source_ontology", "first"),
        surface_forms=("text", lambda s: sorted(set(s))),
        supporting_pmids=("pmid", lambda s: sorted(set(s))),
    )
    .reset_index()
)

print(f"Node table: {len(nodes_df)} unique normalised entities")
nodes_df.head()

Node table: 1002 unique normalised entities


,type,normalized_id,normalized_name,source_ontology,surface_forms,supporting_pmids
0,ChemicalEntity,MeSH:2003432,ISA-2011B,MeSH,[isa-2011b],[41920140]
1,ChemicalEntity,MeSH:2009860,Sildenafil Citrate,MeSH,[sildenafil],[41675105]
2,ChemicalEntity,MeSH:2009892,Imatinib Mesylate,MeSH,[imatinib],[42631458]
3,ChemicalEntity,MeSH:2010026,"Elvitegravir, Cobicistat, Emtricitabine, Tenof...",MeSH,[genvoya],[42633490]
4,ChemicalEntity,MeSH:2012443,BAY 85-8501,MeSH,[bay 85-8501],[41726398]


### Step 6: Attach Normalised IDs Back to Relations

Join `relations_df` to the normalisation lookup on `(pmid, source)` / `(pmid, target)` (via the `matched` table, which already carries type per pmid+text) to get each endpoint's normalised ID.

In [9]:
lookup = resolved_with_pmids[["pmid", "text", "normalized_id"]].drop_duplicates()

relations_normalized = relations_df.merge(
    lookup.rename(columns={"text": "source", "normalized_id": "normalized_source"}),
    on=["pmid", "source"], how="left"
).merge(
    lookup.rename(columns={"text": "target", "normalized_id": "normalized_target"}),
    on=["pmid", "target"], how="left"
)

both_resolved = relations_normalized["normalized_source"].notna() & relations_normalized["normalized_target"].notna()
print(f"Relations with both endpoints resolved: {both_resolved.sum()} / {len(relations_normalized)}")
print(f"Relations with at least one unresolved endpoint (excluded from KG edges below): {(~both_resolved).sum()}")

Relations with both endpoints resolved: 1006 / 1587
Relations with at least one unresolved endpoint (excluded from KG edges below): 581


### Step 7: Build the Edge Table

Group by `(normalized_source, relation, normalized_target)`, aggregating PMIDs into a list -- this is both the final dedup step (post-normalisation, distinct surface forms that resolved to the same entity now correctly merge into one edge) and free provenance (how many/which abstracts support each edge).

In [10]:
clean_relations = relations_normalized[both_resolved].copy()

edges_df = (
    clean_relations
    .groupby(["normalized_source", "relation", "normalized_target"])
    .agg(supporting_pmids=("pmid", lambda s: sorted(set(s))))
    .reset_index()
)
edges_df["n_supporting_abstracts"] = edges_df["supporting_pmids"].apply(len)

print(f"Edge table: {len(edges_df)} unique edges "
      f"(from {len(clean_relations)} relation instances, {len(relations_df)} raw extractions)")
edges_df.sort_values("n_supporting_abstracts", ascending=False).head()

Edge table: 958 unique edges (from 1006 relation instances, 1587 raw extractions)


,normalized_source,relation,normalized_target,supporting_pmids,n_supporting_abstracts
738,NCBIGene:4137,Association,MeSH:68000544,"[42629121, 42629123, 42629124, 42629135]",4
921,NCBITaxon:1773,Association,MeSH:68055985,"[41256823, 42169986, 42632907, 42633156]",4
258,MeSH:68003924,Association,MeSH:68024821,"[42633029, 42633452]",2
244,MeSH:68000544,Association,MeSH:68060825,"[42629124, 42633445]",2
407,MeSH:68053766,Association,MeSH:68000544,"[42629135, 42632811]",2


### Export

`nodes_df` and `edges_df` feed directly into ontology validation and KG construction. `norm_df` (all normalisation attempts, resolved and not) is also exported for appendix/error-analysis reporting.

In [11]:
NEN_DIR = Path(f"{OUTPUT_DIRECTORY}nen")
NEN_DIR.mkdir(parents=True, exist_ok=True)

nodes_df.to_json(NEN_DIR / f"{DATASET_SPLIT}_nodes.json", orient="records", indent=2)
edges_df.to_json(NEN_DIR / f"{DATASET_SPLIT}_edges.json", orient="records", indent=2)
norm_df.to_json(NEN_DIR / f"{DATASET_SPLIT}_normalisation_attempts.json", orient="records", indent=2)

print(f"Saved nodes, edges, and normalisation attempts to '{NEN_DIR}'")

Saved nodes, edges, and normalisation attempts to '..\data\results\final\contemporary\nen'
